# Demo 4 — Rumore bianco e rumore rosa

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
# Rumori riproducibili; stessa RMS digitale, non necessariamente stessa sonorità.
n = 4 * SR
bianco = al.white_noise(n, seed=42)
rosa = al.pink_noise(n, seed=42)
rumori = {}
for file, x in [("white_noise.wav", bianco), ("pink_noise.wav", rosa)]:
    x -= x.mean()
    rumori[file] = 0.1 * x / np.sqrt(np.mean(x ** 2))
fig = make_subplots(rows=2, cols=2, subplot_titles=("rumore bianco", "rumore rosa",
                                              "rumore bianco [dB]", "rumore rosa [dB]"))
for j, (file, nome) in enumerate([("white_noise.wav", "rumore bianco"),
                                  ("pink_noise.wav", "rumore rosa")]):
    y, sr = rumori[file], SR
    f, psd = signal.welch(y, fs=sr, window="hann", nperseg=min(4096, len(y)), scaling="density")
    fig.add_trace(go.Scatter(x=f[1:], y=np.maximum(psd[1:], 1e-16), mode="lines",
                             line=dict(width=1, color="#1769C2"), showlegend=False), row=1, col=j + 1)
    fig.add_trace(go.Scatter(x=f[1:], y=10 * np.log10(np.maximum(psd[1:], 1e-16)), mode="lines",
                             line=dict(width=1, color="#B8860B"), showlegend=False), row=2, col=j + 1)
fig.update_xaxes(type="log", range=[np.log10(20), np.log10(SR / 2)])
fig.update_yaxes(range=[-100, 0], row=2, col=1); fig.update_yaxes(range=[-100, 0], row=2, col=2)
fig.update_layout(height=560, title="Rumore bianco e rosa: densità di potenza lineare e in dB")
fig.show()

for file, nome in [("white_noise.wav", "bianco"), ("pink_noise.wav", "rosa")]:
    y, sr = rumori[file], SR
    banda = al.analysis.feature.spectral_flatness(y=y, n_fft=2048, hop_length=HOP).mean()
    print(f"  {nome:7s} flatness media: {banda:.3f}")

y_b, sr_b = rumori["white_noise.wav"], SR
y_r, sr_r = rumori["pink_noise.wav"], SR
print("rumore bianco"); display(al.safe_audio(al.fade_edges(y_b[:3 * sr_b], sr_b), sr_b, normalize=False))
print("rumore rosa"); display(al.safe_audio(al.fade_edges(y_r[:3 * sr_r], sr_r), sr_r, normalize=False))
